# RAG OCR Spike

Validates vision-model OCR quality on 3 representative pages of a source PDF
**before** committing to the full RAG ingestion pipeline design. Not part of
the production pipeline — a one-off manual test.

Tests the specific layout risks identified for this document type:
- a dense table (row/column alignment)
- body text next to a colored callout box (must stay separate, not interleaved)
- a flyer/template page (non-linear layout, bracketed placeholders)

Assumes the source PDF is already uploaded to a Unity Catalog Volume.

In [ ]:
%pip install pdfplumber
dbutils.library.restartPython()

In [ ]:
# Path to the source PDF inside a Unity Catalog Volume, e.g.
# /Volumes/eliao/wnv_demo/documents/WNV-Outbreak-Communications-Toolkit-2025_508c.pdf
dbutils.widgets.text(
    "pdf_volume_path",
    "/Volumes/eliao/wnv_demo/documents/WNV-Outbreak-Communications-Toolkit-2025_508c.pdf",
)

# Comma-separated 1-indexed page numbers to test.
# Defaults: 3 = callout box, 5 = table, 25 = flyer template.
dbutils.widgets.text("test_pages", "3,5,25")

# Vision-capable Model Serving endpoint to test. Works for either a
# Foundation Model API model (e.g. databricks-gemma-3-12b) or a Databricks
# External Model endpoint you've configured to proxy to a frontier model
# (e.g. Claude, GPT-4o) — both are called the same way via this endpoint name.
dbutils.widgets.text("vision_endpoint", "databricks-gemma-3-12b")

pdf_volume_path = dbutils.widgets.get("pdf_volume_path").strip()
test_pages = [int(p.strip()) for p in dbutils.widgets.get("test_pages").split(",") if p.strip()]
vision_endpoint = dbutils.widgets.get("vision_endpoint").strip()

print(f"PDF: {pdf_volume_path}")
print(f"Test pages: {test_pages}")
print(f"Vision endpoint: {vision_endpoint}")

In [ ]:
import base64
import io
import json
from urllib import request

import pdfplumber

# Notebook-scoped credentials, used only for this test — never stored in
# artifacts, same pattern as test_registered_model.ipynb.
context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
databricks_host = context.apiUrl().get().removeprefix("https://").removesuffix("/")
databricks_token = context.apiToken().get()

EXTRACTION_PROMPT = """\
Extract all text from this document page as clean markdown.

Rules:
- Represent tables as markdown tables, preserving row/column alignment exactly.
- If a colored callout box or sidebar is visually separate from the main
  body text, describe it as a distinct block under a "Callout:" label,
  not interleaved into the paragraph it sits beside.
- Preserve bullet/sub-bullet nesting exactly as shown.
- If the page is a fill-in-the-blank template (contains bracketed
  placeholders like [INSERT ...]), extract it as-is and add a line at the
  top: "TEMPLATE PAGE - contains placeholder fields, not factual content."
- Do not summarize or omit content. Do not invent content not on the page.
"""


def render_page_to_png_bytes(pdf_path: str, page_number: int) -> bytes:
    """Render a 1-indexed PDF page to PNG bytes."""
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_number - 1]
        image = page.to_image(resolution=150)
        buf = io.BytesIO()
        image.original.save(buf, format="PNG")
        return buf.getvalue()


def call_vision_model(image_bytes: bytes) -> str:
    """Send one page image to the configured Databricks vision-capable endpoint."""
    b64 = base64.b64encode(image_bytes).decode()
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": EXTRACTION_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64}"},
                    },
                ],
            }
        ],
        "max_tokens": 2000,
        "temperature": 0.0,
    }
    body = json.dumps(payload).encode()
    url = f"https://{databricks_host}/serving-endpoints/{vision_endpoint}/invocations"
    req = request.Request(
        url,
        data=body,
        headers={
            "Authorization": f"Bearer {databricks_token}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode())
    return result["choices"][0]["message"]["content"]

In [ ]:
results = {}

for page_number in test_pages:
    print(f"\n{'=' * 60}\nPage {page_number}\n{'=' * 60}")
    image_bytes = render_page_to_png_bytes(pdf_volume_path, page_number)
    markdown = call_vision_model(image_bytes)
    results[page_number] = markdown
    print(markdown)

## What to check in the output above

| Page (default) | Pass looks like | Fail looks like |
|---|---|---|
| 5 (table) | All section-header rows correctly identified as spanning rows (not merged into a Type/Description pair); every row correctly paired; nothing shuffled between rows | A section header misread as a real entry; description text bleeding into the wrong row; missing rows |
| 3 (callout box) | The callout box appears as a distinct block, not spliced into the middle of the body paragraph | Callout text interleaved mid-sentence into the body paragraph, or dropped entirely |
| 25 (flyer template) | Correctly flagged as a template; placeholders extracted as-is, not invented; icon/caption pairs stay correctly matched | Model hallucinates real values to fill in placeholders, or garbles which caption belongs to which icon |

If a page fails, try a different `vision_endpoint` (e.g. swap in a frontier
model via a Databricks External Model endpoint) and re-run the cell above —
no other changes needed.